# 第 15 章 · Agentic RL 智能体强化学习

> 📄 配套动画网页：`../15_agent_rl.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第三部分 · 训练

最后一章。前面模型只会「一次说完」，现在要让它学会「多轮行动」：调工具、看结果、再决定下一步。

---


## 🎯 本章目标

- 理解多轮轨迹与单轮回答的区别
- 掌握 `json`、`re` 解析工具调用与答案
- 理解延迟奖励与多组件奖励设计
- 能实现工具调用解析与奖励计算

---


## 📖 背景：奖励不只来自答案

```python
R(τ) = R_answer + R_tool + R_format − R_unfinished
        │          │         │          └ 没给出最终答案要扣分
        │          │         └ 格式错误要扣分
        │          └ 工具调用是否合法、是否有用
        └ 最终答案对不对（可能延迟到最后才结算）
```

与 GRPO 的区别：GRPO 优化一条回答，Agentic RL 优化**一整条轨迹**——
包括「想 → 调工具 → 看结果 → 再想 → 答」的每一步。

**延迟奖励**：调工具这一动作本身可能没有对错，它的价值取决于最终能不能解出题。
所以奖励要等到轨迹结束才结算，再沿着整条轨迹回传。

---



## 🖼 官方配图

![MiniMind 的 RL 结构：训练侧、轨迹侧、rollout 侧解耦](../images/rl-structure.jpg)

*MiniMind 的 RL 结构：训练侧、轨迹侧、rollout 侧解耦*

![Agentic RL 训练曲线](../images/agent_rl_loss.jpg)

*Agentic RL 训练曲线*

![基于 Agentic RL 训练结果的 WebUI：支持思考展示、工具选择与多轮 Tool Use](../images/agent_webui.jpg)

*基于 Agentic RL 训练结果的 WebUI：支持思考展示、工具选择与多轮 Tool Use*

---


# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 15.1 解析工具调用：`json`

In [ ]:
import json

tool_call_text = '''<tool_call>
{"name": "calculator", "arguments": {"expression": "3*12-5"}}
</tool_call>'''

# 提取标签内的 JSON
start = tool_call_text.find("<tool_call>") + len("<tool_call>")
end = tool_call_text.find("</tool_call>")
payload = json.loads(tool_call_text[start:end].strip())
print("工具名  :", payload["name"])
print("参数    :", payload["arguments"])
print("表达式  :", payload["arguments"]["expression"])

### 15.2 正则提取：`re`

In [ ]:
import re

text = "答案是 <answer>31</answer>，用了 <tool_call>calculator</tool_call> 工具。"

ans = re.search(r"<answer>(.*?)</answer>", text)
tools = re.findall(r"<tool_call>(.*?)</tool_call>", text)
print("答案:", ans.group(1) if ans else None)
print("工具:", tools)

### 15.3 校验数学答案

In [ ]:
def check_math(pred: str, gt: str) -> bool:
    def norm(s):
        s = str(s).strip().replace(",", "")
        try:
            return round(float(s), 4)
        except ValueError:
            return s
    return norm(pred) == norm(gt)

print(check_math("31", "31"))          # True
print(check_math("31.0", "31"))        # True（数值归一化）
print(check_math("32", "31"))          # False

### 15.4 安全求值一个简单表达式

真实项目里应该用受限的解析器，这里演示用 `eval` + 白名单的思路。

In [ ]:
import re

def safe_eval(expr: str):
    # 只允许数字和 + - * / ( ) . 空格
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expr):
        return None
    try:
        return eval(expr, {"__builtins__": {}}, {})
    except Exception:
        return None

print("3*12-5 =", safe_eval("3*12-5"))
print("非法输入:", safe_eval("__import__('os').system('ls')"))

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

现在实现智能体的解析与奖励计算。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 15.5 实现 `parse_tool_call`

**要求**：从模型输出文本里提取第一个工具调用，返回 `(name, arguments)`；
如果没有或 JSON 非法，返回 `(None, None)`。

**提示**：先找 `<tool_call>...</tool_call>`，再 `json.loads`，用 `try/except` 兜住解析失败。

In [ ]:
# ======== 你的实现：parse_tool_call ========
#
# 【要求】
#   - 签名：parse_tool_call(text)
#   - 用正则或 find 提取 <tool_call> 与 </tool_call> 之间的内容
#   - json.loads 解析，取出 name 和 arguments
#   - 解析失败或无标签时返回 (None, None)
#
# 【可用到的函数 / 方法】
#   - `re.search`
#   - `json.loads`
#   - `try / except json.JSONDecodeError`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：parse_tool_call ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    text = '<tool_call>{"name": "calculator", "arguments": {"expression": "3*12-5"}}</tool_call>'
    name, args = parse_tool_call(text)
    assert name == "calculator", f"应解析出工具名 calculator，得到 {name}"
    assert args == {"expression": "3*12-5"}, f"参数解析错误，得到 {args}"

    n2, a2 = parse_tool_call("直接回答，没有工具")
    assert n2 is None and a2 is None, f"没有工具调用时应返回 (None, None)，得到 {(n2, a2)}"

    n3, _ = parse_tool_call("<tool_call>{这不是合法 JSON}</tool_call>")
    assert n3 is None, "JSON 非法时应返回 None，而不是抛异常"

check("parse_tool_call 正常 / 缺失 / 非法 JSON", _t)

### 15.6 实现 `compute_reward`

**要求**：给定一条轨迹的文本和标准答案，返回总奖励。

```python
R = R_answer + R_tool + R_format − R_unfinished
```

规则：
- `R_answer`：`<answer>` 里的答案与 `gt_answer` 数值相等给 **1.0**，否则 0
- `R_tool`：解析出至少一个**合法**的 `<tool_call>` 给 **0.3**，否则 0
- `R_format`：有 `<answer>` 标签、且工具调用 JSON 合法给 **0.1**，否则 0
- `R_unfinished`：整条轨迹里没有 `<answer>` 扣 **0.5**

返回 `(total, detail_dict)`，明细 dict 方便调试。

In [ ]:
# ======== 你的实现：compute_reward ========
#
# 【要求】
#   - 签名：compute_reward(text, gt_answer)
#   - 用上面写好的 parse_tool_call 判断有没有合法工具调用
#   - 用 re.search 找 <answer>...</answer>，用 check_math 判对错
#   - R_answer 正确 +1.0；R_tool 有合法工具调用 +0.3
#   - R_format 有 <answer> 且 JSON 合法 +0.1；R_unfinished 没答案 -0.5
#   - 返回 (total, detail_dict)
#
# 【可用到的函数 / 方法】
#   - 上面写好的 `parse_tool_call`
#   - 演示过的 `check_math`（含数值归一化）
#   - `re.search`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：compute_reward ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    good = ('<tool_call>{"name": "calculator", "arguments": {"expression": "3*12-5"}}</tool_call>\n'
            "<answer>31</answer>")
    out = compute_reward(good, "31")
    assert isinstance(out, tuple) and len(out) == 2, "应返回 (总奖励, 明细 dict)"
    total, detail = out
    assert isinstance(detail, dict), "第二个返回值应该是明细 dict"
    close(total, 1.0 + 0.3 + 0.1, tol=1e-6,
          msg="完整且正确的轨迹应得 1.0 + 0.3 + 0.1 = 1.4")

    total2, _ = compute_reward("我不知道", "31")
    assert total2 < 0, f"没给出 <answer> 应扣 R_unfinished，得到 {total2}"

    total3, _ = compute_reward("<answer>32</answer>", "31")
    assert 0 < total3 < 1.0, f"答案错误时不该拿到 R_answer，得到 {total3}"

    total4, _ = compute_reward("<answer>1,000</answer>", "1000")
    assert total4 >= 1.0, f"1,000 与 1000 数值相等应判对，得到 {total4}"

check("compute_reward 各分量", _t)

### 15.7 实现一个简化的单轮 `grpo_step`（进阶）

真实的 Agentic RL 需要与环境交互，比较复杂。这里先用**离线轨迹 + 规则奖励**
把「采样 → 打分 → 归一化 → 更新」这条链路跑通。

**要求**：
1. 对同一个 prompt 用当前策略采样 G 条回答
2. 用 `compute_reward` 给每条打分
3. 用 `compute_advantage`（第 14 章）归一化
4. 用 `grpo_loss`（第 14 章）算损失并反向

**说明**：如果模型没训练好、采不出合法格式，可以先用固定的假 logits 模拟，
重点是把流程走通。

> ⚠️ 这一节是**进阶内容，没有自动测试**——真实的多轮 rollout 需要与环境交互，
> 没法在这里自动验证。请自己打印每条轨迹的奖励、优势与损失，观察它们是否合理。

In [ ]:
# ======== 你的实现：grpo_step ========
#
# 【要求】
#   - 签名：grpo_step(model, tokenizer, prompt, G=4, ...)
#   - 采样 G 条回答（model.generate 或 Categorical 采样）
#   - 对每条算 compute_reward，得到 rewards [G]
#   - advantages = compute_advantage(rewards)
#   - 算 logp_new / logp_old，调用 grpo_loss，反向 + 更新
#
# 【可用到的函数 / 方法】
#   - 第 14 章的 `compute_advantage` 和 `grpo_loss`
#   - `torch.distributions.Categorical`
#   - `optimizer.step()` / `zero_grad()`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
summary()

## 🎓 结语

到这里，你已经把 MiniMind 的**全部零件**亲手实现了一遍：

- **模型**：RMSNorm → RoPE → 注意力/GQA → SwiGLU → MoE → Block → 完整模型
- **训练**：预训练 → SFT → LoRA → 蒸馏 → DPO → GRPO/PPO/CISPO → Agentic RL

**下一步建议**：
1. 把这 15 章所有留白代码块都自己写一遍；
2. 对照官方 `model/model_minimind.py` 逐行 diff，看差异在哪；
3. 改一个超参（层数 / head 数 / 词表大小）重新训练，观察 loss 与生成质量的变化；
4. 把小模型部署起来（`web_demo.py`），亲自和它对话。

# 第三部分 · 自测与思考

- [ ] 能说清多轮轨迹与单轮回答在奖励设计上的区别
- [ ] 「延迟奖励」为什么必要？
- [ ] `compute_reward` 的四个分量各自防止了什么坏行为？
- [ ] 整个项目的 15 个知识点，你能独立讲出来几个？

**思考题**：如果只保留 `R_answer` 而不要 `R_format`，模型会学到什么奇怪的输出格式？
